# Maps: 21 cm and 3.46 cm brightness temperature, neutral fraction

From the **production** maps (`Tb_maps_prod/H_real`, `Tb_maps_prod/He`, 640³, real space) and the 640³ renders (`HII_VolumeFraction`).
The figure code is that of `visual_maps.ipynb` cells 7, 8 and 13 (that notebook is kept as the archive); only the data sources changed
(and a redundant first `imshow` in the 21 cm cell). Each panel is one 640³ plane (0.78 cMpc thick) at index N/2 of the third axis.

Notes for the captions: the bottom colour of the 21 cm bar ("0") holds every pixel below 0.1 mK (ionized gas, 10⁻⁶–10⁻² mK; most of the
panels at z ≲ 6) and the top colour everything above 100 mK (star-forming haloes, up to ~2000 mK); at z ≲ 6 the remaining H I is mostly
star-forming gas (neutral by the render convention). The 3.46 cm bar's bottom colour (10⁻⁶) holds every pixel below 10⁻⁶ µK. At z ≤ 5 the
640³ 3.46 cm maps are qualitative (cell-level runs: amplitude ×4–7 low, large-scale pattern lost at z = 4; JALPHA_NOTES §6.2).


In [ ]:
import os
import h5py
import glob
import numpy as np
import pandas as pd
import math
from scipy.integrate import quad
from astropy import constants as const
from scipy.interpolate import interp1d
from scipy.interpolate import CubicSpline

from scipy.signal import savgol_filter

import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 200
mpl.rcParams['savefig.dpi'] = 300

import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.family": "STIXGeneral",
    "mathtext.fontset": "stix",
    "font.size": 14,
})
import cmasher as cmr
import gc
gc.collect()

import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
import re

import seaborn as sns
from matplotlib.colors import SymLogNorm, LogNorm, Normalize
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.patheffects as path_effects

# =====================================
# Production delta T_b maps (real space; run_21cm.sh / run_3cm.sh). File names are not zero-padded.
# =====================================
MAPS = {"H":  "/orcd/data/mvogelsb/004/chcho/Tb_maps_prod/H_real",
        "He": "/orcd/data/mvogelsb/004/chcho/Tb_maps_prod/He"}


## 21 cm δT_b (visual_maps.ipynb cell 7)


In [ ]:
from matplotlib.ticker import LogLocator, NullLocator


# =====================================
# Helper
# =====================================

def get_redshift(ps_file):
    with open(ps_file) as f:
        return float(f.readline().strip())

#/orcd/data/mvogelsb/004/chcho/compute_J_alpha_H/Tb_H
# =====================================
# Input files
# =====================================
snaps = [156, 181, 209, 255, 289, 325, 364, 406]


# =====================================
# Figure layout
# =====================================

fig = plt.figure(figsize=(12, 6))

gs = gridspec.GridSpec(
    2, 4,
    wspace=0,
    hspace=0
)

axes = [
    fig.add_subplot(gs[i, j])
    for i in range(2)
    for j in range(4)
]


# =====================================
# Color normalization
# Tb map with symlog including zero
# =====================================
"""
norm = SymLogNorm(
    linthresh=1,
    linscale=1,
    vmin=-10,
    vmax=100,
    base=10
)
"""
#norm = TwoSlopeNorm(vmin=-10, vcenter=0, vmax=100)
#norm = TwoSlopeNorm(vmin=-10, vcenter=0, vmax=50)
#norm = TwoSlopeNorm(vmin=-0.1, vcenter=10, vmax=100)
#norm = TwoSlopeNorm(vmin=0, vcenter=10, vmax=100)
norm = LogNorm(vmin=0.1, vmax=100)

# =====================================
# Load + plot hydrogen Tb maps
# =====================================

for i, snap in enumerate(snaps):

    hdf5_file = f"{MAPS['H']}/RIN_{snap}.hdf5"

    PS_file = f"{MAPS['H']}/PS_{snap}.out"

    z = get_redshift(PS_file)

    # ---------------------------------
    # Load cube
    # ---------------------------------

    with h5py.File(hdf5_file, "r") as f:
        Tb_cube = f["/rin"][:]

    N = Tb_cube.shape[0]

    # Mid-plane slice
    Tb = Tb_cube[:, :, N // 2]

    ax = axes[i]

    im = ax.imshow(
        np.where(Tb > 0, Tb, np.nan).T,
        origin="lower",
        cmap=sns.color_palette("managua", as_cmap=True),
        norm=norm,
        extent=[0, 500, 0, 500]
    )
    
    # Negative overlay — dark spots
    ax.imshow(
        np.where(Tb < 0, Tb, np.nan).T,
        origin="lower",
        cmap="Reds_r",
        norm=Normalize(vmin=-10, vmax=0),
        extent=[0, 500, 0, 500]
    )
   
#    im = ax.imshow(
#        Tb.T,
#        origin="lower",
#        cmap=cmr.pride,   # pass directly, no sns.color_palette wrapper
#        norm=norm,
#        extent=[0, 500, 0, 500]
#    )

    # Remove ticks
    ax.set_xticks([])
    ax.set_yticks([])

    # =================================
    # Better redshift label
    # (from new figure style)
    # =================================

    txt = ax.text(
        0.95,
        0.95,
        fr"$z = {z:.1f}$",
        transform=ax.transAxes,
        color="white",
        fontsize=15,
        ha="right",
        va="top"
    )

    txt.set_path_effects([
        path_effects.Stroke(
            linewidth=3,
            foreground="0.2"
        ),
        path_effects.Normal()
    ])


# =====================================
# Inset colorbar + scale bar
# in last panel
# =====================================

ax_last = axes[-1]

cax = inset_axes(
    ax_last,
    width="40%",
    height="5%",
    loc="lower right",
    #borderpad=2.7
    bbox_to_anchor=(-0.1, 0.22, 1, 1),   # (x, y, w, h) in axes fraction; increase x to move right
    bbox_transform=ax_last.transAxes,
    borderpad=0
)

# THIS creates the actual bar
cbar = fig.colorbar(
    im,
    cax=cax,
    orientation="horizontal"
)

# Tick locations

cbar.set_ticks([0.1, 1, 10, 100])
cbar.set_ticklabels(["0", "1", "10", "100"])
cbar.ax.xaxis.set_minor_locator(NullLocator())  # remove minor ticks
#ticks
#cbar.set_ticks([-10, -1, 0, 1, 10, 100])
#cbar.set_ticks([-10, 0, 50, 100])
#cbar.set_ticklabels(["-10", "-1", "0", "1", "10", "100"])


# =====================================
# Make colorbar texts visible using
# outline (same style as redshift text)
# =====================================



# -------------------------------------
# Tick labels with outline
# -------------------------------------

#cbar.ax.tick_params(
#    labelsize=12,
#    colors="black"   # white text
#)

cbar.ax.tick_params(labelsize=12, colors="black")


#for tick in cbar.ax.get_xticklabels():
#    tick.set_color("black")
#    tick.set_fontweight("bold")
    
#for tick in cbar.ax.get_xticklabels():
#    tick.set_color("white")
#    tick.set_path_effects([
#        path_effects.Stroke(linewidth=1.5, foreground="black"),
#        path_effects.Normal()
#    ])

#for tick in cbar.ax.get_xticklabels():
#    tick.set_color("white")
#    tick.set_path_effects([
#        path_effects.Stroke(
#            linewidth=2,
#            foreground="black"
#        ),
#        path_effects.Normal()
#    ])

# -------------------------------------
# TOP label = physical scale bar
# -------------------------------------

#title = cbar.ax.set_title(
#    "200 cMpc",
#    fontsize=11,
#    pad=4,
#    color="black"
#)

#title.set_path_effects([
#    path_effects.Stroke(
#        linewidth=2,
#        foreground="black"
#    ),
#    path_effects.Normal()
#])

# -------------------------------------
# BOTTOM label = Tb quantity
# -------------------------------------


label = cbar.ax.text(
    0.5, -2.1,
    #r"$\delta T_b^{\mathrm{H}}\ [{\rm mK}]$",
    r"$\boldsymbol{\delta T_b^{\mathrm{H}}}\ \mathbf{[mK]}$",
    transform=cbar.ax.transAxes,
    fontsize=12, ha="center", va="top",
    color="black"
)
#label.set_path_effects([
#    path_effects.Stroke(linewidth=2.0, foreground="black"),
#    path_effects.Normal()
#])

#label.set_path_effects([
#    path_effects.Stroke(
#        linewidth=2,
#        foreground="black"
#    ),
#    path_effects.Normal()
#])
# =====================================
# Scale bar: 100 cMpc on the upper-left panel
# =====================================

ax_first = axes[0]

# Bar geometry in data coordinates (extent is 0-500 cMpc)
bar_length = 100.0          # cMpc
x0 = 40                     # left end of bar (data units)
y0 = 460                    # height of bar (near top)

# The bar itself
ax_first.plot(
    [x0, x0 + bar_length],
    [y0, y0],
    color="white",
    lw=3,
    solid_capstyle="butt",
    path_effects=[
        path_effects.Stroke(linewidth=5, foreground="0.15"),
        path_effects.Normal()
    ]
)

# Label above the bar
lbl = ax_first.text(
    x0 + bar_length / 2,
    y0 - 35,
    r"$100\ \mathrm{cMpc}$",
    color="white",
    fontsize=12,
    ha="center",
    va="top"
)
lbl.set_path_effects([
    path_effects.Stroke(linewidth=3, foreground="0.2"),
    path_effects.Normal()
])
# =====================================
# Final layout
# =====================================

#plt.tight_layout()

# plt.savefig(
#     "Tb_H_map_symlog_inset.png",
#     dpi=300,
#     bbox_inches="tight"
# )

plt.show()

## Neutral fraction x_HI = 1 − HII_VolumeFraction (cell 8)


In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.patheffects as path_effects
# -----------------------------------
# Color scale
#
# We plot the neutral hydrogen fraction, x_HI (= 1 - x_HII), on a
# standard logarithmic color scale (LogNorm) from 1 (neutral) down to
# 10^-N_DECADES (highly ionized). This is the same transform as
# before, just expressed as what it actually is -- a log scale on
# x_HI -- instead of a custom forward/inverse pair on x_HII. Pixel
# colors are identical either way; this version is the one you can
# describe in a methods section without hand-waving: "colored by the
# neutral hydrogen fraction, x_HI, on a base-10 log scale from 1 to
# 1e-N_DECADES."
#
# CMAP is reversed ("magma_r") because we're now coloring by the
# NEUTRAL fraction: low neutral fraction (= highly ionized) should be
# bright, high neutral fraction (= neutral gas) should be black --
# magma_r puts black at its high end, matching that.
#
# N_DECADES: how many decades of neutral fraction to resolve before
# saturating.
#   4 -> saturates at 1-x_HII = 1e-4  (x_HII = 0.9999)
#   6 -> saturates at 1-x_HII = 1e-6  (x_HII = 0.999999)  [confirmed good]
# -----------------------------------
N_DECADES = 6
CMAP      = "magma_r"
VERBOSE   = True   # print percentiles of (1 - x_HII) so N_DECADES can be tuned
EPS = 10.0 ** (-N_DECADES)
norm = LogNorm(vmin=EPS, vmax=1.0)
# ---- real log-scale colorbar ticks, in neutral-fraction (x_HI) units ----
CBAR_TICKS  = [1e-6, 1e-3, 1.0]
CBAR_LABELS = [r"$10^{-6}$", r"$10^{-3}$", "1"]
# -----------------------------------
# Helper
# -----------------------------------
def get_redshift(ps_file):
    with open(ps_file) as f:
        return float(f.readline().strip())
def hdf5_path(snap):
    if snap <= 428:
        run = "Flagship_new"
    else:
        run = "Flagship_below_z_4p75_new"
    return (
        f"/orcd/data/mvogelsb/005/mkilab001/"
        f"mvogelsblab003/Lab/Thesan-XL/"
        f"{run}/postprocessing/ren_640/"
        f"HII_VolumeFraction/"
        f"HII_VolumeFraction_{snap:03d}.hdf5"
    )
# -----------------------------------
# SNAPSHOTS
# -----------------------------------
snaps = [156, 181, 209, 255, 289, 325, 364, 406]
# -----------------------------------
# FIGURE LAYOUT
# -----------------------------------
fig = plt.figure(figsize=(12, 6))
gs = gridspec.GridSpec(
    2, 4,
    wspace=0,
    hspace=0
)
axes = [
    fig.add_subplot(gs[i, j])
    for i in range(2)
    for j in range(4)
]
# -----------------------------------
# LOAD + PLOT HII MAPS
# -----------------------------------
for i, snap in enumerate(snaps):
    PS_file = (
        f"{MAPS['H']}/PS_{snap}.out"
    )
    z = get_redshift(PS_file)
    # ---- mid-plane slice, read directly (no full cube in memory) ----
    with h5py.File(hdf5_path(snap), "r") as f:
        dset = f["/HII_VolumeFraction"]
        N = dset.shape[0]
        f_HII = dset[:, :, N // 2]
    f_HII_plot = np.clip(f_HII, 0.0, 1.0)
    x_HI = np.clip(1.0 - f_HII_plot, EPS, 1.0)  # the field actually plotted
    if VERBOSE:
        p = np.percentile(x_HI, [1, 25, 50, 75, 99])
        print(
            f"snap {snap:3d}  z={z:5.2f}  "
            f"x_HI percentiles [1,25,50,75,99]: "
            + "  ".join(f"{v:.2e}" for v in p),
            flush=True
        )
    # ---- plot ----
    ax = axes[i]
    im = ax.imshow(
        x_HI.T,
        origin="lower",
        cmap=CMAP,
        norm=norm,
        extent=[0, 500, 0, 500]
    )
    ax.set_xticks([])
    ax.set_yticks([])
    # ---- redshift label ----
    txt = ax.text(
        0.95,
        0.95,
        fr"$z = {z:.1f}$",
        transform=ax.transAxes,
        color="white",
        fontsize=15,
        ha="right",
        va="top"
    )
    txt.set_path_effects([
        path_effects.Stroke(
            linewidth=3,
            foreground="0.2"
        ),
        path_effects.Normal()
    ])
# -----------------------------------
# Inset colorbar inside last panel
# -----------------------------------
ax_last = axes[-1]
cax = inset_axes(
    ax_last,
    width="40%",
    height="5%",
    loc="lower right",
    bbox_to_anchor=(-0.1, 0.22, 1, 1),   # (x, y, w, h) in axes fraction
    bbox_transform=ax_last.transAxes,
    borderpad=0
)
cbar = fig.colorbar(
    im,
    cax=cax,
    orientation="horizontal"
)
cbar.set_ticks(CBAR_TICKS)
cbar.set_ticklabels(CBAR_LABELS)
cbar.ax.tick_params(labelsize=12, colors="white")
cbar.ax.xaxis.set_minor_locator(plt.NullLocator())
for t in cbar.ax.get_xticklabels():
    t.set_path_effects([
        path_effects.Stroke(linewidth=2, foreground="0.2"),
        path_effects.Normal()
    ])
# BOTTOM label = neutral hydrogen fraction, x_HI (= 1 - x_HII)
label = cbar.ax.text(
    0.5,
    -2.1,
    r"$x_{\mathrm{H \ I}}$",
    transform=cbar.ax.transAxes,
    fontsize=12,
    ha="center",
    va="top",
    color="white"
)
label.set_path_effects([
    path_effects.Stroke(linewidth=2, foreground="0.2"),
    path_effects.Normal()
])
# -----------------------------------
# Scale bar: 100 cMpc on the upper-left panel
# -----------------------------------
ax_first = axes[0]
bar_length = 100.0          # cMpc
x0 = 40                     # left end of bar (data units)
y0 = 460                    # height of bar (near top)
ax_first.plot(
    [x0, x0 + bar_length],
    [y0, y0],
    color="white",
    lw=3,
    solid_capstyle="butt",
    path_effects=[
        path_effects.Stroke(linewidth=5, foreground="0.15"),
        path_effects.Normal()
    ]
)
lbl = ax_first.text(
    x0 + bar_length / 2,
    y0 - 35,
    r"$100\ \mathrm{cMpc}$",
    color="white",
    fontsize=12,
    ha="center",
    va="top"
)
lbl.set_path_effects([
    path_effects.Stroke(linewidth=3, foreground="0.2"),
    path_effects.Normal()
])
# -----------------------------------
# SAVE + SHOW
# -----------------------------------
# plt.savefig(
#     "f_HII_map_log_inset.png",
#     dpi=300,
#     bbox_inches="tight"
# )
plt.show()

## 3.46 cm δT_b (cell 13)


In [ ]:

# =====================================
# Helper
# =====================================

def get_redshift(ps_file):
    with open(ps_file) as f:
        return float(f.readline().strip())


# =====================================
# Input files
# =====================================
snaps = [209, 255, 325, 406, 452, 501, 582, 705]


#ps_files = [f"./TS_TK/He/Tb/PS_Tb_He_{snap}.out" for snap in snaps]

# =====================================
# Figure layout (better publication size)
# =====================================

fig = plt.figure(figsize=(12, 6))

gs = gridspec.GridSpec(
    2, 4,
    wspace=0,
    hspace=0
)

axes = [
    fig.add_subplot(gs[i, j])
    for i in range(2)
    for j in range(4)
]


# =====================================
# Color normalization
# =====================================
"""
norm = SymLogNorm(
    linthresh=.1,
    linscale=1,
    vmin=0,
    vmax=10, #vmax=100,
    base=10
)
"""


norm = LogNorm(vmin=1e-6, vmax=10)

# =====================================
# Load + plot helium Tb maps
# =====================================

for i, snap in enumerate(snaps):

    hdf5_file = f"{MAPS['He']}/RIN_{snap}.hdf5"

    PS_file = f"{MAPS['He']}/PS_{snap}.out"

    z = get_redshift(PS_file)

    # ---------------------------------
    # Load cube
    # ---------------------------------

    with h5py.File(hdf5_file, "r") as f:
        Tb_cube = f["/rin"][:]

    N = Tb_cube.shape[0]
    Tb = Tb_cube[:, :, N // 2]

    ax = axes[i]

    im = ax.imshow(
        Tb.T,
        origin="lower",
        cmap=sns.color_palette(
            "berlin_r",
            as_cmap=True
        ),
        norm=norm,
        extent=[0, 500, 0, 500]
    )


    #    im = ax.imshow(
#        Tb.T,
#        origin="lower",
#        cmap=sns.color_palette(
#            "seismic", #"rocket_r, "winter", "berlin", "managua"
#            as_cmap=True
#        ),
#        norm=norm,
#        extent=[0, 500, 0, 500]
#    )

    # remove ticks
    ax.set_xticks([])
    ax.set_yticks([])

    # =================================
    # Redshift label with boundary
    # =================================

    txt = ax.text(
        0.95,
        0.95,
        fr"$z = {z:.1f}$",
        transform=ax.transAxes,
        color="white",
        fontsize=15,
        ha="right",
        va="top"
    )

    txt.set_path_effects([
        path_effects.Stroke(
            linewidth=3,
            foreground="0.2"
        ),
        path_effects.Normal()
    ])


# =====================================
# Inset colorbar + scale bar
# in last panel
# =====================================

ax_last = axes[-1]

cax = inset_axes(
    ax_last,
    width="40%",
    height="5%",
    loc="lower right",
    #borderpad=2.7
    bbox_to_anchor=(-0.1, 0.22, 1, 1),   # (x, y, w, h) in axes fraction; increase x to move right
    bbox_transform=ax_last.transAxes,
    borderpad=0
)

# THIS creates the actual bar
cbar = fig.colorbar(
    im,
    cax=cax,
    orientation="horizontal"
)

cbar.set_ticks([1e-6, 1e-2, 1e1])
cbar.set_ticklabels([r"$10^{-6}$",r"$10^{-2}$", r"$10^{1}$"])
"""

# Tick locations
cbar.set_ticks([
    0,
    1,
    10,
    100
])

cbar.set_ticklabels([
    "0",
    "1",
    "10",
    "100"
])
"""

# =====================================
# Make colorbar texts visible using
# outline (same style as redshift text)
# =====================================

import matplotlib.patheffects as path_effects

# -------------------------------------
# Tick labels with outline
# -------------------------------------

cbar.ax.tick_params(
    labelsize=12,
    colors="white"   # white text
)

#for tick in cbar.ax.get_xticklabels():
#    tick.set_color("white")
#    tick.set_path_effects([
#        path_effects.Stroke(
#            linewidth=2,
#            foreground="black"
#        ),
#        path_effects.Normal()
#    ])

# -------------------------------------
# TOP label = physical scale bar
# -------------------------------------


#title.set_path_effects([
#    path_effects.Stroke(
#        linewidth=2,
#        foreground="black"
#    ),
#    path_effects.Normal()
#])

# -------------------------------------
# BOTTOM label = Tb quantity
# -------------------------------------

label = cbar.ax.text(
    0.5,
    -2.1,
    r"$\boldsymbol{\delta T_b^{\mathrm{He}}}\ \mathbf{[\mu K]}$",
    transform=cbar.ax.transAxes,
    fontsize=12,
    ha="center",
    va="top",
    color="white"
)



ax_first = axes[0]

# Bar geometry in data coordinates (extent is 0-500 cMpc)
bar_length = 100.0          # cMpc
x0 = 40                     # left end of bar (data units)
y0 = 460                    # height of bar (near top)

# The bar itself
ax_first.plot(
    [x0, x0 + bar_length],
    [y0, y0],
    color="white",
    lw=3,
    solid_capstyle="butt",
    path_effects=[
        path_effects.Stroke(linewidth=5, foreground="0.15"),
        path_effects.Normal()
    ]
)

# Label above the bar
lbl = ax_first.text(
    x0 + bar_length / 2,
    y0 - 35,
    r"$100\ \mathrm{cMpc}$",
    color="white",
    fontsize=12,
    ha="center",
    va="top"
)
lbl.set_path_effects([
    path_effects.Stroke(linewidth=3, foreground="0.2"),
    path_effects.Normal()
])

# =====================================
# Final layout
# =====================================

# plt.savefig(
#     "Tb_He_map_inset.png",
#     dpi=300,
#     bbox_inches="tight"
# )

plt.show()